In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# Phase 6 — Final Model Training & Stability Validation

**Project:** Wholesale Customer Segmentation

Phase 6 - Model Training & Validation
Wholesale Customers Clustering Analysis

Steps covered (per implementation plan):
 16.   Train final K-Means model (fix random_state, n_init)
 17.   Assign cluster labels
 17.5. Sanity-check cluster stability

Depends on: phase1_setup.py (df_raw), phase2_eda.py (SPEND_COLS, FIG_DIR),
            phase4_transform_scale.py output (scaled_features.csv),
            phase5_algorithm_selection.py (K selection: K=2 statistical optimum,
            K=3 and K=4 retained as plausible more-granular alternatives)
Outputs: PNG figures saved to ./figures/, labeled_customers.csv, stability_summary.csv, printed diagnostics

Assumption (explicitly stated, carried from Phase 5):
 Phase 6 compares K=2, K=3, and K=4 so the statistical optimum and nearby
more-granular alternatives can be evaluated for robustness before the
primary solution is finalized.

### How to use this notebook
Run cells from top to bottom. Keep the project files in the same folder as this notebook. Phases 1–4 create the data preparation artifacts used by later phases; Phases 5–9 read those artifacts; Phase 10 assembles the final report.

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.utils import resample



# Recreate upstream constants/data locally so this notebook is standalone.
DATA_PATH = "data/raw/wholesale_customers.csv"
df_raw = pd.read_csv(DATA_PATH)
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
FIG_DIR = "figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_style("whitegrid")

RANDOM_STATE = 42
N_INIT = 20
CANDIDATE_KS = [2, 3, 4]           # carried from Phase 5
STABILITY_SEEDS = [42, 7, 123, 2024, 99]   # 5 different seeds for stability check
N_BOOTSTRAP = 50                # bootstrap resamples for stability check
STABILITY_THRESHOLD = 0.75       # project-defined practical heuristic threshold

## SECTION 1: Load Scaled Features & Raw Data

In [ ]:
# SECTION 1: Load Scaled Features & Raw Data
# ===========================================================================
def load_data() -> tuple[pd.DataFrame, np.ndarray]:
    """Load the scaled feature matrix (model input) and confirm alignment
    with df_raw (used later to attach labels back to untransformed data)."""
    print("=" * 70)
    print("SECTION 1: LOAD SCALED FEATURES & RAW DATA")
    print("=" * 70)

    scaled_df = pd.read_csv("scaled_features.csv")
    print(f"Scaled feature matrix shape: {scaled_df.shape}")
    print(f"df_raw shape: {df_raw.shape}")

    assert len(scaled_df) == len(df_raw), (
        "Row count mismatch between scaled_features.csv and df_raw — "
        "cannot safely assign labels back without matching row order."
    )
    print("[OK] Row counts match and are index-aligned (same row order preserved "
          "since Phase 4 never sorted/filtered df_raw).")

    return scaled_df, scaled_df.values

## SECTION 2: Train Final K-Means Model(s) (Step 16)

In [ ]:
# SECTION 2: Train Final K-Means Model(s) (Step 16)
# ===========================================================================
def train_final_models(X: np.ndarray) -> dict:
    """Train the final K-Means model for each candidate K, with a fixed
    random_state and n_init for full reproducibility."""
    print("\n" + "=" * 70)
    print("SECTION 2: TRAIN FINAL K-MEANS MODEL(S)")
    print("=" * 70)
    print(f"Fixed settings: random_state={RANDOM_STATE}, n_init={N_INIT!r}")

    models = {}
    for k in CANDIDATE_KS:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
        labels = km.fit_predict(X)
        models[k] = {"model": km, "labels": labels}
        sizes = pd.Series(labels).value_counts().sort_index()
        print(f"\nK={k}: trained. Cluster sizes: {sizes.to_dict()} "
              f"(inertia={km.inertia_:.2f})")

    print("\n[OK] Final model(s) trained with locked random_state/n_init — "
          "re-running this script reproduces identical cluster assignments.")
    return models

## SECTION 3: Assign Cluster Labels Back to Raw (Untransformed) Data (Step 17)

In [ ]:
# SECTION 3: Assign Cluster Labels Back to Raw (Untransformed) Data (Step 17)
# ===========================================================================
def assign_labels_to_raw(models: dict) -> pd.DataFrame:
    """Attach each candidate model's cluster labels to a copy of df_raw, so
    downstream profiling (Phase 7) interprets clusters using original,
    human-readable spend values rather than scaled/log-transformed ones."""
    print("\n" + "=" * 70)
    print("SECTION 3: ASSIGN CLUSTER LABELS TO RAW DATA")
    print("=" * 70)

    df_labeled = df_raw.copy()  # df_raw itself remains untouched (Phase 1 rule)
    for k, result in models.items():
        col_name = f"cluster_k{k}"
        df_labeled[col_name] = result["labels"]
        print(f"Added column '{col_name}' — value counts: "
              f"{df_labeled[col_name].value_counts().sort_index().to_dict()}")

    print(f"\nLabeled dataframe shape: {df_labeled.shape}")
    print(df_labeled.head().to_string())
    return df_labeled

## SECTION 4: Stability Check — Multiple Random Seeds (Step 17.5)

In [ ]:
# SECTION 4: Stability Check — Multiple Random Seeds (Step 17.5)
# ===========================================================================
def check_stability_seeds(X: np.ndarray, k: int) -> pd.DataFrame:
    """Re-run K-Means with several different random_state seeds and compute
    pairwise Adjusted Rand Index (ARI) between runs. ARI close to 1.0 means
    the clustering solution is stable regardless of initialization; ARI
    close to 0 means the solution found is not robust."""
    print(f"\n--- Seed-variation stability check for K={k} ---")

    seed_labels = {}
    for seed in STABILITY_SEEDS:
        km = KMeans(n_clusters=k, random_state=seed, n_init=N_INIT)
        seed_labels[seed] = km.fit_predict(X)

    rows = []
    seeds = list(seed_labels.keys())
    for i in range(len(seeds)):
        for j in range(i + 1, len(seeds)):
            ari = adjusted_rand_score(seed_labels[seeds[i]], seed_labels[seeds[j]])
            rows.append({"seed_a": seeds[i], "seed_b": seeds[j], "ari": ari})

    ari_df = pd.DataFrame(rows)
    print(ari_df.round(4).to_string(index=False))
    print(f"Mean pairwise ARI across {len(STABILITY_SEEDS)} seeds: {ari_df['ari'].mean():.4f} "
          f"(min={ari_df['ari'].min():.4f}, max={ari_df['ari'].max():.4f})")
    return ari_df

## SECTION 5: Stability Check — Bootstrap Resampling (Step 17.5)

In [ ]:
# SECTION 5: Stability Check — Bootstrap Resampling (Step 17.5)
# ===========================================================================
def check_stability_bootstrap(X: np.ndarray, k: int) -> pd.DataFrame:
    """Re-run K-Means on bootstrap resamples of the data (sampling with
    replacement) and compare each resample's clustering, restricted to the
    common out-of-resample rows, against the full-data model via ARI.
    This tests whether the cluster structure is an artifact of specific
    rows or reflects a genuinely stable pattern in the population."""
    print(f"\n--- Bootstrap stability check for K={k} ({N_BOOTSTRAP} resamples) ---")

    baseline_km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
    baseline_labels_full = baseline_km.fit_predict(X)

    n = X.shape[0]
    rows = []
    for b in range(N_BOOTSTRAP):
        boot_idx = resample(np.arange(n), replace=True, n_samples=n, random_state=b)
        oob_idx = np.setdiff1d(np.arange(n), np.unique(boot_idx))  # out-of-bag rows
        if len(oob_idx) < k * 2:
            continue  # skip if too few OOB points to meaningfully compare

        boot_km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
        boot_km.fit(X[boot_idx])

        # Predict OOB rows using both the bootstrap model and the baseline
        # (full-data) model, then compare their agreement on those rows.
        boot_pred_oob = boot_km.predict(X[oob_idx])
        baseline_pred_oob = baseline_labels_full[oob_idx]
        ari = adjusted_rand_score(baseline_pred_oob, boot_pred_oob)
        rows.append({"bootstrap_run": b, "n_oob": len(oob_idx), "ari_vs_baseline": ari})

    boot_df = pd.DataFrame(rows)
    print(boot_df.round(4).to_string(index=False))
    print(f"Mean ARI (bootstrap models vs. baseline, on out-of-bag rows): "
          f"{boot_df['ari_vs_baseline'].mean():.4f} "
          f"(min={boot_df['ari_vs_baseline'].min():.4f}, max={boot_df['ari_vs_baseline'].max():.4f})")
    return boot_df

## SECTION 6: Visualize Stability Results

In [ ]:
# SECTION 6: Visualize Stability Results
# ===========================================================================
def plot_stability_summary(stability_results: dict) -> None:
    """Bar chart comparing mean ARI (seed-variation and bootstrap) across
    candidate K values, to visually support the stability conclusion."""
    print("\n" + "=" * 70)
    print("SECTION 6: VISUALIZE STABILITY RESULTS")
    print("=" * 70)

    ks = list(stability_results.keys())
    seed_means = [stability_results[k]["seed_ari"]["ari"].mean() for k in ks]
    boot_means = [stability_results[k]["boot_ari"]["ari_vs_baseline"].mean() for k in ks]

    x = np.arange(len(ks))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width / 2, seed_means, width, label="Seed-variation ARI", color="steelblue")
    ax.bar(x + width / 2, boot_means, width, label="Bootstrap ARI", color="darkorange")
    ax.axhline(0.75, color="gray", linestyle="--", linewidth=1, label="Common 'stable' threshold (0.75)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"K={k}" for k in ks])
    ax.set_ylabel("Mean Adjusted Rand Index")
    ax.set_title("Cluster Stability: Mean ARI by K (Seed-Variation vs. Bootstrap)")
    ax.set_ylim(0, 1.05)
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/13_stability_ari_summary.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/13_stability_ari_summary.png")

## SECTION 7: Document Stability Conclusion (Step 17.5)

In [ ]:
# SECTION 7: Document Stability Conclusion (Step 17.5)
# ===========================================================================
def document_stability_conclusion(stability_results: dict) -> str:
    """Summarize whether each candidate K's clustering is stable enough to
    trust for business interpretation in Phase 7, using a standard ARI
    stability threshold."""
    print("\n" + "=" * 70)
    print("SECTION 7: DOCUMENT STABILITY CONCLUSION")
    print("=" * 70)

    STABILITY_THRESHOLD = 0.75  # project-defined practical heuristic threshold
    lines = [f"Stability threshold used: mean ARI >= {STABILITY_THRESHOLD} = stable (project-defined heuristic)\n"]

    for k, res in stability_results.items():
        seed_mean = res["seed_ari"]["ari"].mean()
        boot_mean = res["boot_ari"]["ari_vs_baseline"].mean()
        seed_ok = seed_mean >= STABILITY_THRESHOLD
        boot_ok = boot_mean >= STABILITY_THRESHOLD
        verdict = "STABLE" if (seed_ok and boot_ok) else (
            "PARTIALLY STABLE" if (seed_ok or boot_ok) else "UNSTABLE")
        lines.append(
            f"K={k}: seed-variation ARI={seed_mean:.4f} ({'OK' if seed_ok else 'BELOW THRESHOLD'}), "
            f"bootstrap ARI={boot_mean:.4f} ({'OK' if boot_ok else 'BELOW THRESHOLD'}) "
            f"-> VERDICT: {verdict}"
        )

    conclusion = "\n".join(lines)
    print(conclusion)
    print(
        "\n[NOTE] This stability check validates that clusters are not "
        "initialization artifacts or sensitive to which customers happen to "
        "be in the sample — a necessary precondition before drawing business "
        "conclusions from cluster membership in Phase 7."
    )

    # Synthesis across the key K candidates.
    stable = {}
    for k, res in stability_results.items():
        stable[k] = (
            res["seed_ari"]["ari"].mean() >= STABILITY_THRESHOLD
            and res["boot_ari"]["ari_vs_baseline"].mean() >= STABILITY_THRESHOLD
        )

    if 2 in stability_results and 3 in stability_results and 4 in stability_results:
        conclusion += (
            "\nSYNTHESIS ACROSS K=2, K=3, AND K=4:\n"
            "  K=2 is the primary candidate because it has the highest silhouette score in Phase 5 "
            "and is checked here for robustness against initialization and bootstrap perturbations.\n"
            f"  Seed-variation ARI: K=2={stability_results[2]['seed_ari']['ari'].mean():.2f}, "
            f"K=3={stability_results[3]['seed_ari']['ari'].mean():.2f}, "
            f"K=4={stability_results[4]['seed_ari']['ari'].mean():.2f}.\n"
            f"  Bootstrap ARI: K=2={stability_results[2]['boot_ari']['ari_vs_baseline'].mean():.2f}, "
            f"K=3={stability_results[3]['boot_ari']['ari_vs_baseline'].mean():.2f}, "
            f"K=4={stability_results[4]['boot_ari']['ari_vs_baseline'].mean():.2f}.\n"
            "  These stability results are interpreted together with silhouette, elbow behavior, "
            "hierarchical clustering, and business interpretability.\n"
        )

    return conclusion


def save_stability_summary(stability_results: dict) -> None:
    rows = []
    for k, res in stability_results.items():
        seed_mean = res["seed_ari"]["ari"].mean()
        boot_mean = res["boot_ari"]["ari_vs_baseline"].mean()
        verdict = "Stable" if seed_mean >= STABILITY_THRESHOLD and boot_mean >= STABILITY_THRESHOLD else "Not stable"
        rows.append({"K": k, "Seed ARI": seed_mean, "Bootstrap ARI": boot_mean, "Verdict": verdict})
    pd.DataFrame(rows).to_csv("stability_summary.csv", index=False)
    print("[OK] Stability summary saved to stability_summary.csv")

## SECTION 8: Persist Labeled Data for Downstream Phases

In [ ]:
# SECTION 8: Persist Labeled Data for Downstream Phases
# ===========================================================================
def save_labeled_data(df_labeled: pd.DataFrame) -> None:
    """Save the raw data with cluster labels attached, for Phase 7 profiling."""
    print("\n" + "=" * 70)
    print("SECTION 8: PERSIST LABELED DATA")
    print("=" * 70)

    out_path = "labeled_customers.csv"
    df_labeled.to_csv(out_path, index=False)
    print(f"[OK] Saved {out_path} (shape: {df_labeled.shape}) — contains original "
          "Channel/Region/spend columns plus 'cluster_k2' and 'cluster_k3' "
          "label columns for Phase 7 profiling.")

## MAIN — run Phase 6 end to end

In [ ]:
# MAIN — run Phase 6 end to end
# ===========================================================================
if __name__ == "__main__":
    scaled_df, X = load_data()
    models = train_final_models(X)
    df_labeled = assign_labels_to_raw(models)

    stability_results = {}
    for k in CANDIDATE_KS:
        seed_ari = check_stability_seeds(X, k)
        boot_ari = check_stability_bootstrap(X, k)
        stability_results[k] = {"seed_ari": seed_ari, "boot_ari": boot_ari}

    plot_stability_summary(stability_results)
    document_stability_conclusion(stability_results)
    save_labeled_data(df_labeled)
    save_stability_summary(stability_results)

    print("\n" + "=" * 70)
    print("PHASE 6 COMPLETE")
    print("=" * 70)
    print(f"[OK] Final K-Means model(s) trained for K={CANDIDATE_KS} with fixed "
          f"random_state={RANDOM_STATE}, n_init={N_INIT!r}.")
    print("[OK] Cluster labels assigned back to untransformed df_raw copy.")
    print("[OK] Stability checked via 5-seed variation and 50-run bootstrap, "
          "measured with Adjusted Rand Index.")
    print("[OK] Labeled data saved to labeled_customers.csv.")
    print("[OK] Ready for Phase 7 (Cluster Profiling & Business Analysis).")

### Phase 6 checkpoint

Review the outputs and figures generated by this phase before moving to the next phase.